# JSON Inputs Setup (Before vs After Change)

This notebook starts by loading two JSON files:

- **Before change** from `JSON Whole Model/Ifc2x3_Duplex_Architecture.json`
- **After change** from `JSON_Edit/Ifc2x3_Duplex_Architecture.json`

The loaded variables are:
- `json_before_data`
- `json_after_data`

In [1]:
import json
from pathlib import Path

file_name = "Ifc2x3_Duplex_Architecture.json"

# Resolve workspace root whether running from repo root or notebook folder.
base_dir = Path.cwd()
if not (base_dir / "JSON_Edit").exists() and (base_dir.parent / "JSON_Edit").exists():
    base_dir = base_dir.parent

json_before_path = base_dir / "JSON Whole Model" / file_name
json_after_path = base_dir / "JSON_Edit" / file_name

with json_before_path.open("r", encoding="utf-8") as f:
    json_before_data = json.load(f)

with json_after_path.open("r", encoding="utf-8") as f:
    json_after_data = json.load(f)

print(f"Loaded BEFORE JSON: {json_before_path}")
print(f"Loaded AFTER JSON:  {json_after_path}")

Loaded BEFORE JSON: c:\Git\APS-IFC\JSON Whole Model\Ifc2x3_Duplex_Architecture.json
Loaded AFTER JSON:  c:\Git\APS-IFC\JSON_Edit\Ifc2x3_Duplex_Architecture.json


In [2]:
before_count = len(json_before_data) if isinstance(json_before_data, list) else 0
after_count = len(json_after_data) if isinstance(json_after_data, list) else 0

print(f"Before-change records: {before_count}")
print(f"After-change records:  {after_count}")

if before_count > 0:
    print("Sample BEFORE item keys:", list(json_before_data[0].keys())[:10])
if after_count > 0:
    print("Sample AFTER item keys:", list(json_after_data[0].keys())[:10])

Before-change records: 1432
After-change records:  1432
Sample BEFORE item keys: ['Name', 'DbId', 'ExternalId', 'Properties']
Sample AFTER item keys: ['Name', 'DbId', 'ExternalId', 'Properties']


## Revised IFC COBie Comparison Table

This section compares **revised (after-change)** IFC-related COBie parameter values against the **before-change** JSON and prints only changed/new rows.

Included IFC structure categories:
- `IFC`
- `IFCMATERIALLAYER`
- `IFCMATERIALLAYERSETUAGE`

In [9]:
import pandas as pd
from IPython.display import display


def _norm(v):
    if v is None:
        return ""
    return str(v).strip()


def _build_key(item):
    dbid = item.get("DbId")
    external_id = _norm(item.get("ExternalId"))
    if dbid is not None and external_id:
        return f"DbId:{dbid}|ExternalId:{external_id}"
    if dbid is not None:
        return f"DbId:{dbid}"
    if external_id:
        return f"ExternalId:{external_id}"
    return None


TARGET_CATEGORIES = {"ifc", "ifcmateriallayer", "ifcmateriallayersetuage"}


def _extract_item_type(item):
    props = item.get("Properties", [])
    if not isinstance(props, list):
        return ""

    for prop in props:
        if not isinstance(prop, dict):
            continue

        category = _norm(prop.get("category")).lower()
        display_name = _norm(prop.get("displayName")).lower()
        value = _norm(prop.get("value"))
        if category == "item" and display_name == "type":
            return value

    return ""


def _extract_cobie_props(item):
    """Return IFC-related COBie properties with non-empty values from one item."""
    result = []
    props = item.get("Properties", [])
    if not isinstance(props, list):
        return result

    for p in props:
        if not isinstance(p, dict):
            continue

        category = _norm(p.get("category"))
        display_name = _norm(p.get("displayName"))
        value = _norm(p.get("value"))

        if category.lower() in TARGET_CATEGORIES and value:
            haystack = f"{category} {display_name} {value}".lower()
            if "cobie" in haystack:
                result.append(
                    {
                        "category": category,
                        "parameter": display_name,
                        "value": value,
                    }
                )

    return result


before_map = {}
for obj in json_before_data:
    if isinstance(obj, dict):
        k = _build_key(obj)
        if k:
            before_map[k] = obj

after_map = {}
for obj in json_after_data:
    if isinstance(obj, dict):
        k = _build_key(obj)
        if k:
            after_map[k] = obj


rows = []
for k, after_obj in after_map.items():
    before_obj = before_map.get(k, {})

    after_props = _extract_cobie_props(after_obj)
    if not after_props:
        # Only include revised objects that currently carry COBie values.
        continue

    before_props = _extract_cobie_props(before_obj) if isinstance(before_obj, dict) else []
    before_lookup = {(x["category"], x["parameter"]): x["value"] for x in before_props}

    revised_item_type = _extract_item_type(after_obj)
    before_item_type = _extract_item_type(before_obj) if isinstance(before_obj, dict) else ""
    detected_object_type = revised_item_type or before_item_type or "Unknown"

    for ap in after_props:
        pair_key = (ap["category"], ap["parameter"])
        before_value = _norm(before_lookup.get(pair_key))
        after_value = _norm(ap["value"])

        if before_value != after_value:
            change_type = "Added" if not before_value and after_value else "Updated"
            rows.append(
                {
                    "Object Type": detected_object_type,
                    "DbId": after_obj.get("DbId"),
                    "ExternalId": _norm(after_obj.get("ExternalId")),
                    "Name (Before)": _norm(before_obj.get("Name")) if isinstance(before_obj, dict) else "",
                    "Name (Revised)": _norm(after_obj.get("Name")),
                    "IFC Structure": ap["category"],
                    "COBie Parameter": ap["parameter"],
                    "Before Value": before_value,
                    "Revised Value": after_value,
                    "Change Type": change_type,
                }
            )

revised_ifc_cobie_changes_df = pd.DataFrame(rows)

if revised_ifc_cobie_changes_df.empty:
    print("No revised IFC COBie value changes were found compared to before JSON.")
else:
    revised_ifc_cobie_changes_df = revised_ifc_cobie_changes_df.sort_values(
        by=["Object Type", "IFC Structure", "COBie Parameter", "Name (Revised)", "DbId"], kind="stable"
    ).reset_index(drop=True)

    print(f"Revised IFC COBie changed rows: {len(revised_ifc_cobie_changes_df)}")
    print("Changed rows by object type:")
    display(
        revised_ifc_cobie_changes_df.groupby(["Object Type", "Change Type"], dropna=False)
        .size()
        .rename("Rows")
        .reset_index()
        .sort_values(by=["Object Type", "Change Type"], kind="stable")
        .reset_index(drop=True)
    )
    display(revised_ifc_cobie_changes_df)

Revised IFC COBie changed rows: 231
Changed rows by object type:


,Object Type,Change Type,Rows
0,IFCDOOR,Added,14
1,IFCFOOTING,Added,7
2,IFCFURNISHINGELEMENT,Added,10
3,IFCSPACE,Added,11
4,IFCWALL,Added,1
5,IFCWALLSTANDARDCASE,Added,56
6,IFCWINDOW,Added,22
7,LcIFCRepresentationHolder,Added,110


,Object Type,DbId,ExternalId,Name (Before),Name (Revised),IFC Structure,COBie Parameter,Before Value,Revised Value,Change Type
0,IFCDOOR,41,0/0/0/0/32,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150173,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150173,IFC,COBie,,EF_25_30_25 : Doors,Added
1,IFCDOOR,42,0/0/0/0/33,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150257,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150257,IFC,COBie,,EF_25_30_25 : Doors,Added
2,IFCDOOR,742,0/0/0/1/68,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:203720,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:203720,IFC,COBie,,EF_25_30_25 : Doors,Added
3,IFCDOOR,744,0/0/0/1/70,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:204034,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:204034,IFC,COBie,,EF_25_30_25 : Doors,Added
4,IFCDOOR,707,0/0/0/1/33,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150378,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150378,IFC,COBie,,EF_25_30_25 : Doors,Added
...,...,...,...,...,...,...,...,...,...,...
226,LcIFCRepresentationHolder,1176,0/0/0/1/54/0,Plan,Plan,IFC,COBie,,EF_25_10 : Walls,Added
227,LcIFCRepresentationHolder,1186,0/0/0/1/55/0,Plan,Plan,IFC,COBie,,EF_25_10 : Walls,Added
228,LcIFCRepresentationHolder,1196,0/0/0/1/56/0,Plan,Plan,IFC,COBie,,EF_25_10 : Walls,Added
229,LcIFCRepresentationHolder,1232,0/0/0/1/68/0,Plan,Plan,IFC,COBie,,EF_25_10 : Walls,Added
